In [1]:
from pathlib import Path

print("Working directory:", Path.cwd())
print("Data folder exists:", Path("data").exists())

Working directory: /home/student/work
Data folder exists: True


In [2]:
import sys
import pyspark

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("PySpark version:", pyspark.__version__)

Python executable: /opt/conda/bin/python
Python version: 3.10.19 | packaged by conda-forge | (main, Jan 26 2026, 23:45:08) [GCC 14.3.0]
PySpark version: 3.5.5


In [3]:
%pip install kafka-python==3.0.11 pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 614.2/614.2 kB 20.5 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pyarrow
from kafka import KafkaProducer
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler

print("Required imports succeeded.")

Required imports succeeded.


In [1]:
import os
import sys
import pyspark


connector = (
    "org.apache.spark:"
    "spark-sql-kafka-0-10_2.12:3.5.5"
)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f"--driver-memory 2g --packages {connector} pyspark-shell"
)

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ITO5202 Assessment 2")
    .master("local[2]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.session.timeZone", "America/New_York")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark:", spark.version)
print(
    "Scala:",
    spark._jvm.scala.util.Properties.versionNumberString()
)
print("Master:", spark.sparkContext.master)
print(
    "Driver memory:",
    spark.sparkContext.getConf().get("spark.driver.memory")
)
print("Timezone:", spark.conf.get("spark.sql.session.timeZone"))

assert spark.version == pyspark.__version__

spark.range(3).show()

Spark: 3.5.5
Scala: 2.12.18
Master: local[2]
Driver memory: 2g
Timezone: America/New_York
+---+
| id|
+---+
|  0|
|  1|
|  2|
+---+



In [2]:
import json
from kafka import KafkaProducer

producer = KafkaProducer(
    bootstrap_servers=["kafka:9092"]
)

try:
    payload = [{"status": "environment_ready"}]

    result = producer.send(
        "a2_setup_test",
        value=json.dumps(payload).encode("utf-8")
    ).get(timeout=20)

    print("Topic:", result.topic)
    print("Partition:", result.partition)
    print("Offset:", result.offset)
finally:
    producer.close()

Topic: a2_setup_test
Partition: 0
Offset: 0


In [3]:
test_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", "a2_setup_test")
    .option("startingOffsets", "earliest")
    .load()
    .selectExpr("CAST(value AS STRING) AS payload")
)

query = (
    test_stream.writeStream
    .format("memory")
    .queryName("a2_environment_check")
    .outputMode("append")
    .start()
)

try:
    query.processAllAvailable()

    spark.sql(
        "SELECT payload FROM a2_environment_check"
    ).show(truncate=False)
finally:
    query.stop()

+---------------------------------+
|payload                          |
+---------------------------------+
|[{"status": "environment_ready"}]|
+---------------------------------+



In [4]:
import subprocess
import sys
from pathlib import Path

packages = subprocess.check_output(
    [sys.executable, "-m", "pip", "freeze"],
    text=True
)

requirements_path = Path("/home/student/work/requirements.txt")
requirements_path.write_text(packages, encoding="utf-8")

print("Python:", sys.version.split()[0])
print("Created:", requirements_path)

Python: 3.10.19
Created: /home/student/work/requirements.txt


In [5]:
from pathlib import Path

data_dir = Path("/home/student/work/data")
data_dir.mkdir(parents=True, exist_ok=True)

required_files = [
    f"yellow_tripdata_2023-{month:02d}.parquet"
    for month in range(1, 13)
] + ["taxi_zone_lookup.csv"]

available_files = {
    path.name: path
    for path in data_dir.rglob("*")
    if path.is_file()
}

for name in required_files:
    if name in available_files:
        print("FOUND:", available_files[name].relative_to(data_dir))
    else:
        print("MISSING:", name)

FOUND: yellow_tripdata_2023-01.parquet
FOUND: yellow_tripdata_2023-02.parquet
FOUND: yellow_tripdata_2023-03.parquet
FOUND: yellow_tripdata_2023-04.parquet
FOUND: yellow_tripdata_2023-05.parquet
FOUND: yellow_tripdata_2023-06.parquet
FOUND: yellow_tripdata_2023-07.parquet
FOUND: yellow_tripdata_2023-08.parquet
FOUND: yellow_tripdata_2023-09.parquet
FOUND: yellow_tripdata_2023-10.parquet
FOUND: yellow_tripdata_2023-11.parquet
FOUND: yellow_tripdata_2023-12.parquet
FOUND: taxi_zone_lookup.csv
